# Strands SDK: Interrupt System (Human-in-the-Loop) Tutorial

This notebook walks through the **interrupt** system — the SDK's mechanism for pausing an agent to ask a human for input before continuing.

## What You'll Learn

1. What the core interrupt classes are and how they work
2. How to trigger an interrupt from a **hook** (before a tool runs)
3. How to trigger an interrupt from a **tool** (during execution)
4. How to respond to interrupts and resume the agent
5. What happens inside the SDK at each step

## Prerequisites

- AWS credentials configured (for Amazon Bedrock)
- Strands SDK installed (`pip install strands-agents strands-agents-builder`)
- Read `00-python-basics-for-interrupts.md` if you're new to Python concepts like exceptions, dataclasses, and protocols

## Quick Concept Recap

| Concept | What it is | Role in interrupt system |
|---------|-----------|-------------------------|
| **Exception** | An error you can `raise` and `catch` with `try/except` | `InterruptException` stops execution when human input is needed |
| **Dataclass** | A class that holds data (like a struct) | `Interrupt` holds id, name, reason, response |
| **Protocol** | An interface — "must have these methods" | `_Interruptible` shares `interrupt()` between hooks and tools |
| **TypedDict** | A dictionary with known, typed keys | `InterruptResponse` structures the user's answer |

See `00-python-basics-for-interrupts.md` for detailed explanations with examples.

## Part 1: Exploring the Core Classes

Before running any agents, let's import and inspect the building blocks.

In [1]:
# === Imports ===
# We import the core interrupt classes directly from the SDK source

from strands.interrupt import Interrupt, InterruptException, _InterruptState
from strands.types.interrupt import _Interruptible, InterruptResponse, InterruptResponseContent

print("Imports successful!")

Imports successful!


### The `Interrupt` Dataclass

An `Interrupt` is like a **question card**. It has:
- `id` — a unique identifier (auto-generated by the SDK)
- `name` — a name you choose (e.g., `"for_delete_tool"`)
- `reason` — why you're interrupting (e.g., `"need approval"`)
- `response` — the user's answer (starts as `None`, filled in later)

**Source:** `src/strands/interrupt.py:11-29`

In [2]:
# Create an Interrupt manually to see what it looks like
demo_interrupt = Interrupt(
    id="demo-123",          # Normally auto-generated, we set it manually here
    name="for_delete_tool", # A name we pick to identify this interrupt
    reason="need approval", # Why we're asking the user
    response=None           # No answer yet — user hasn't responded
)

# Print each field to see the structure
print(f"id:       {demo_interrupt.id}")
print(f"name:     {demo_interrupt.name}")
print(f"reason:   {demo_interrupt.reason}")
print(f"response: {demo_interrupt.response}")  # None — nobody answered yet
print()

# to_dict() serializes it (useful for saving state)
print(f"As dict: {demo_interrupt.to_dict()}")

id:       demo-123
name:     for_delete_tool
reason:   need approval
response: None

As dict: {'id': 'demo-123', 'name': 'for_delete_tool', 'reason': 'need approval', 'response': None}


### The `InterruptException`

This is a custom Python exception. When raised, it **stops execution**. The SDK catches it (never crashes) and extracts the `Interrupt` object from inside.

Think of it as throwing a ball with a note attached. The catcher reads the note.

**Source:** `src/strands/interrupt.py:32-37`

In [ ]:
# Create an InterruptException with our demo interrupt inside
exc = InterruptException(demo_interrupt)

# The exception carries the Interrupt object
print(f"Exception type: {type(exc).__name__}")
print(f"Interrupt inside: {exc.interrupt}")
print()

# Demonstrate the raise/catch flow — this is exactly what happens inside the SDK
try:
    # This simulates what happens when interrupt() is called and response is None
    raise InterruptException(demo_interrupt)

except InterruptException as caught:
    # This simulates what the hook registry or tool decorator does
    print(f"Caught an InterruptException!")
    print(f"  Interrupt name: {caught.interrupt.name}")
    print(f"  Reason: {caught.interrupt.reason}")
    print(f"  Response: {caught.interrupt.response}")
    print()
    print("The SDK collects this interrupt and returns it to you.")

### The `_InterruptState` — Saved Game State

This tracks whether the agent is paused and stores the context needed to resume.

- `activated` — is the agent currently paused?
- `interrupts` — dictionary of pending interrupts
- `context` — saved execution state (which tool was running, partial results)

**Source:** `src/strands/interrupt.py:40-120`

In [ ]:
# Create a fresh interrupt state
state = _InterruptState()
print(f"Initial state:")
print(f"  activated: {state.activated}")    # False — not paused
print(f"  interrupts: {state.interrupts}")  # Empty — no pending questions
print(f"  context: {state.context}")        # Empty — nothing saved
print()

# Simulate what happens when an interrupt is raised:
# 1. Add the interrupt to the state
state.interrupts["demo-123"] = demo_interrupt

# 2. Save the execution context (tool message, partial results)
state.context = {"tool_use_message": "<model's tool request>", "tool_results": []}

# 3. Activate (pause the agent)
state.activate()

print(f"After interrupt:")
print(f"  activated: {state.activated}")    # True — agent is paused
print(f"  interrupts: {list(state.interrupts.keys())}")  # Has our interrupt
print(f"  context keys: {list(state.context.keys())}")   # Has saved state
print()

# Simulate deactivation (after successful resume)
state.deactivate()
print(f"After deactivate:")
print(f"  activated: {state.activated}")    # False — back to normal
print(f"  interrupts: {state.interrupts}")  # Cleared
print(f"  context: {state.context}")        # Cleared

### The Dual-Call Trick: How `interrupt()` Works

The `interrupt()` method on `_Interruptible` is the heart of the system. It runs **twice**:

1. **First call** — `response` is `None` → raises `InterruptException` (agent pauses)
2. **Second call** — `response` is `"APPROVE"` → returns the response (agent continues)

Let's look at the actual source code:

In [ ]:
import inspect

# Print the actual source code of the interrupt() method
source = inspect.getsource(_Interruptible.interrupt)
print(source)

In [ ]:
# Let's simulate the dual-call trick manually

# Create a fresh state (like a fresh agent)
state = _InterruptState()
interrupt_id = "test-interrupt-001"

# === FIRST CALL (no response yet) ===
print("=== First call (no response) ===")

# setdefault: creates a NEW Interrupt because the key doesn't exist yet
interrupt_obj = state.interrupts.setdefault(
    interrupt_id,
    Interrupt(id=interrupt_id, name="confirm", reason="need approval", response=None)
)

print(f"  Interrupt response: {interrupt_obj.response}")  # None

if interrupt_obj.response is not None:
    print(f"  Would return: {interrupt_obj.response}")
else:
    print(f"  Response is None -> would raise InterruptException")
    print(f"  Agent PAUSES here.")

print()

# === USER RESPONDS (simulating resume()) ===
print("=== User responds ===")
state.interrupts[interrupt_id].response = "APPROVE"  # User filled in the answer
print(f"  Set response to: {state.interrupts[interrupt_id].response}")

print()

# === SECOND CALL (response exists now) ===
print("=== Second call (response exists) ===")

# setdefault: finds EXISTING Interrupt (does NOT overwrite)
interrupt_obj = state.interrupts.setdefault(
    interrupt_id,
    Interrupt(id=interrupt_id, name="confirm", reason="need approval", response=None)
)

print(f"  Interrupt response: {interrupt_obj.response}")  # "APPROVE"

if interrupt_obj.response is not None:
    print(f"  Would return: {interrupt_obj.response}")
    print(f"  Agent CONTINUES with the user's answer.")
else:
    print(f"  Would raise InterruptException")

---

## Part 2: Hook-Based Interrupt (Complete Working Example)

Now let's run a real agent with a real interrupt. In this example:

1. We define a `delete_tool` that deletes things
2. We create a hook that **interrupts before** `delete_tool` runs
3. The agent pauses and asks us for approval
4. We respond, and the agent continues

### How Hooks Work

A **hook** is code that runs at specific points in the agent's lifecycle. `BeforeToolCallEvent` runs right before a tool is called. By registering a callback for this event, we can inspect (or block) any tool call.

**Source:** `src/strands/hooks/events.py:119-155`

In [ ]:
from typing import Any
from strands import Agent, tool
from strands.hooks import BeforeToolCallEvent, HookProvider, HookRegistry
from strands.models.bedrock import BedrockModel

print("Imports ready.")

In [ ]:
# Define a simple tool that "deletes" something.
# The @tool decorator registers this function as a tool the agent can call.

@tool
def delete_tool(key: str) -> str:
    """Delete an object by its key."""
    # In a real app, this might delete a file, database record, etc.
    return f"Deleted object with key '{key}'"

print(f"Tool defined: {delete_tool.tool_name}")

In [ ]:
# Define a hook that interrupts before delete_tool runs.
#
# HookProvider is a base class for organizing hooks.
# register_hooks() tells the SDK which events we care about.
# The callback (approve) runs every time a tool is about to be called.

class DeleteApprovalHook(HookProvider):

    def register_hooks(self, registry: HookRegistry, **kwargs: Any) -> None:
        # Register our callback for the BeforeToolCallEvent.
        # This means: "call self.approve() before any tool runs."
        registry.add_callback(BeforeToolCallEvent, self.approve)

    def approve(self, event: BeforeToolCallEvent) -> None:
        # event.tool_use contains info about which tool is being called.
        # We only want to interrupt for delete_tool, not other tools.
        if event.tool_use["name"] != "delete_tool":
            return  # Not delete_tool, let it proceed without interruption

        # Call event.interrupt() — this is the key line!
        # First time: raises InterruptException (agent pauses)
        # Second time (after user responds): returns the user's answer
        approval = event.interrupt(
            "delete_approval",          # Name for this interrupt
            reason="Approve deletion?"   # Reason shown to the user
        )

        # This line only runs on the SECOND call (after user responded).
        # On the first call, the exception was raised above and we never get here.
        print(f"User responded with: {approval}")

        if approval != "APPROVE":
            # User didn't approve — cancel the tool.
            # cancel_tool tells the SDK to skip this tool and return this message instead.
            event.cancel_tool = "Deletion was not approved by user."

print("Hook defined: DeleteApprovalHook")

In [ ]:
# Create the agent with our hook and tool.
#
# hooks=[DeleteApprovalHook()] — registers our interrupt hook
# tools=[delete_tool]          — gives the agent access to delete_tool
# callback_handler=None        — disables streaming output (cleaner for this demo)

model = BedrockModel()
agent = Agent(
    model=model,
    hooks=[DeleteApprovalHook()],
    tools=[delete_tool],
    system_prompt="You delete objects given their keys. Always use the delete_tool.",
    callback_handler=None,
)

print("Agent created with DeleteApprovalHook.")

### Triggering the Interrupt

When we call the agent, it will:
1. Send our request to the model
2. Model decides to call `delete_tool`
3. `BeforeToolCallEvent` fires → our hook runs → `event.interrupt()` raises `InterruptException`
4. Agent pauses and returns an `AgentResult` with `stop_reason="interrupt"`

In [ ]:
# Call the agent — this will trigger the interrupt
result = agent("Please delete the object with key 'my-file-123'")

# Check the result
print(f"Stop reason: {result.stop_reason}")  # Should be "interrupt"
print(f"Number of interrupts: {len(result.interrupts)}")
print()

# Examine each interrupt
for i, interrupt in enumerate(result.interrupts):
    print(f"Interrupt #{i + 1}:")
    print(f"  id:       {interrupt.id}")        # Auto-generated unique ID
    print(f"  name:     {interrupt.name}")       # "delete_approval" (what we set)
    print(f"  reason:   {interrupt.reason}")     # "Approve deletion?" (what we set)
    print(f"  response: {interrupt.response}")   # None — user hasn't answered yet

### Responding to the Interrupt

Now we build a response and call the agent again. The response format is:
```python
{"interruptResponse": {"interruptId": "<the interrupt's id>", "response": "<your answer>"}}
```

The agent will:
1. Call `resume()` to fill in `interrupt.response`
2. Skip the model (it already has the tool request saved)
3. Re-run the hook — this time `event.interrupt()` returns `"APPROVE"`
4. The tool runs, model generates final answer

In [ ]:
# Build the response — we approve the deletion
responses = []
for interrupt in result.interrupts:
    responses.append({
        "interruptResponse": {
            "interruptId": interrupt.id,   # Must match the interrupt's ID exactly
            "response": "APPROVE"           # Our answer — the hook checks for this
        }
    })

print(f"Sending {len(responses)} response(s):")
for r in responses:
    print(f"  ID: {r['interruptResponse']['interruptId'][:40]}...")
    print(f"  Response: {r['interruptResponse']['response']}")
print()

# Resume the agent with our responses
result = agent(responses)

print(f"Stop reason: {result.stop_reason}")  # Should be "end_turn" (normal completion)
print(f"Agent response: {result}")

### What If We Reject?

Let's run it again, but this time respond with something other than `"APPROVE"`. The hook will set `event.cancel_tool`, which cancels the tool call.

In [ ]:
# Start a fresh agent to avoid conversation history interference
agent_reject = Agent(
    model=model,
    hooks=[DeleteApprovalHook()],
    tools=[delete_tool],
    system_prompt="You delete objects given their keys. Always use the delete_tool.",
    callback_handler=None,
)

# Trigger the interrupt
result = agent_reject("Delete the object with key 'important-data'")
print(f"Stop reason: {result.stop_reason}")  # "interrupt"
print()

# This time, respond with "REJECT" (anything other than "APPROVE")
responses = [
    {
        "interruptResponse": {
            "interruptId": result.interrupts[0].id,
            "response": "REJECT"  # Not "APPROVE" — hook will cancel the tool
        }
    }
]

result = agent_reject(responses)
print(f"Stop reason: {result.stop_reason}")
print(f"Agent response: {result}")
# The agent should report that the deletion was not approved

---

## Part 3: Tool-Based Interrupt (Using ToolContext)

Instead of interrupting **before** a tool runs (via a hook), you can interrupt **during** tool execution using `ToolContext`.

This is useful when the tool itself needs to ask the user something — for example, confirming a specific value or choosing between options.

### Key Differences

| | Hook-based | Tool-based |
|---|-----------|------------|
| **Where** | Before tool runs | During tool execution |
| **How** | `event.interrupt()` on `BeforeToolCallEvent` | `tool_context.interrupt()` on `ToolContext` |
| **Use case** | Approval gates, access control | Mid-execution decisions, user input |

**Source:** `src/strands/types/tools.py:129-160`

In [ ]:
from strands.types.tools import ToolContext

# Define a tool that interrupts mid-execution to ask the user for a timezone.
#
# context=True tells the SDK to inject a ToolContext object as the first argument.
# ToolContext has the interrupt() method (it implements _Interruptible).

@tool(name="schedule_meeting", context=True)
def schedule_meeting(tool_context: ToolContext, title: str, time: str) -> str:
    """Schedule a meeting with a title and time."""

    # Ask the user which timezone to use.
    # First call: raises InterruptException (agent pauses)
    # Second call (after user responds): returns their answer
    timezone = tool_context.interrupt(
        "timezone_selection",                          # Interrupt name
        reason=f"Which timezone for '{title}' at {time}?"  # Question for user
    )

    # This line only executes on the second call (after user responded)
    return f"Meeting '{title}' scheduled at {time} {timezone}"

print(f"Tool defined: {schedule_meeting.tool_name}")

In [ ]:
# Create an agent with the schedule_meeting tool (no hooks needed this time)
agent_tool = Agent(
    model=model,
    tools=[schedule_meeting],
    system_prompt="You schedule meetings. Always use the schedule_meeting tool.",
    callback_handler=None,
)

# Call the agent — the tool will interrupt to ask for timezone
result = agent_tool("Schedule a team standup at 9:00 AM")

print(f"Stop reason: {result.stop_reason}")  # "interrupt"
print(f"Number of interrupts: {len(result.interrupts)}")
print()

for interrupt in result.interrupts:
    print(f"Interrupt:")
    print(f"  name:   {interrupt.name}")    # "timezone_selection"
    print(f"  reason: {interrupt.reason}")  # "Which timezone for 'team standup' at 9:00 AM?"
    print(f"  response: {interrupt.response}")  # None

In [ ]:
# Respond with a timezone
responses = [
    {
        "interruptResponse": {
            "interruptId": result.interrupts[0].id,
            "response": "PST"  # Our answer: Pacific Standard Time
        }
    }
]

# Resume the agent
result = agent_tool(responses)

print(f"Stop reason: {result.stop_reason}")  # "end_turn"
print(f"Agent response: {result}")
# Should mention the meeting was scheduled at 9:00 AM PST

---

## Part 4: What Happened Inside?

Let's trace through the internal flow step by step. We'll enable logging and re-run to see the SDK's internal operations.

### The Internal Flow

```
YOU                          AGENT                        HOOK/TOOL
 |                             |                              |
 |-- "delete X" ------------->|                              |
 |                             |-- model says: use tool ----->|
 |                             |                              |
 |                             |                   interrupt() called
 |                             |                   response=None -> RAISE
 |                             |<---- InterruptException -----|
 |                             |                              |
 |                             | saves state, activates       |
 |<-- result.interrupts -------|                              |
 |                             |                              |
 | (user decides)              |                              |
 |                             |                              |
 |-- [interruptResponse] ---->|                              |
 |                             | resume() fills response      |
 |                             |-- re-runs tool execution --->|
 |                             |                              |
 |                             |                   interrupt() called
 |                             |                   response="APPROVE" -> RETURN
 |                             |                   tool continues normally
 |                             |<---- tool result -------------|
 |                             |                              |
 |                             | deactivates, continues loop  |
 |<-- final answer ------------|                              |
```

### Key Source Locations

| Step | File | Line | What happens |
|------|------|------|-------------|
| interrupt() raises | `types/interrupt.py` | 111 | `raise InterruptException(interrupt_)` |
| Hook catches it | `hooks/registry.py` | 238 | `except InterruptException as exception` |
| Tool catches it | `tools/decorator.py` | 609 | `except InterruptException as e` |
| Event loop saves state | `event_loop/event_loop.py` | 487 | `agent._interrupt_state.context = {...}` |
| Event loop activates | `event_loop/event_loop.py` | 488 | `agent._interrupt_state.activate()` |
| Resume fills response | `interrupt.py` | 100 | `self.interrupts[interrupt_id].response = ...` |
| Event loop skips model | `event_loop/event_loop.py` | 144 | `if agent._interrupt_state.activated` |
| Partial results restored | `event_loop/event_loop.py` | 461 | `tool_results.extend(...)` |
| interrupt() returns | `types/interrupt.py` | 109 | `return interrupt_.response` |
| State cleared | `event_loop/event_loop.py` | 505 | `agent._interrupt_state.deactivate()` |

In [ ]:
import logging

# Enable DEBUG logging for the event loop to see internal operations
logging.basicConfig(level=logging.DEBUG, format='%(name)s - %(message)s')

# Only show strands logs, not all libraries
logging.getLogger('strands').setLevel(logging.DEBUG)
logging.getLogger('botocore').setLevel(logging.WARNING)
logging.getLogger('urllib3').setLevel(logging.WARNING)

# Create a fresh agent
agent_debug = Agent(
    model=model,
    hooks=[DeleteApprovalHook()],
    tools=[delete_tool],
    system_prompt="You delete objects given their keys. Always use the delete_tool.",
    callback_handler=None,
)

print("=" * 60)
print("PHASE 1: Triggering the interrupt")
print("=" * 60)
result = agent_debug("Delete object with key 'test-key'")
print(f"\nResult: stop_reason={result.stop_reason}")
print(f"Interrupts: {[i.name for i in result.interrupts]}")

In [ ]:
print("=" * 60)
print("PHASE 2: Resuming with approval")
print("=" * 60)

responses = [
    {
        "interruptResponse": {
            "interruptId": result.interrupts[0].id,
            "response": "APPROVE"
        }
    }
]

result = agent_debug(responses)
print(f"\nResult: stop_reason={result.stop_reason}")
print(f"Response: {result}")

# Reset logging level
logging.getLogger('strands').setLevel(logging.WARNING)

---

## Summary

### The Interrupt System in One Paragraph

The interrupt system lets you **pause an agent** during tool execution to ask a human for input. It works by raising an `InterruptException` (a custom Python exception) that the SDK catches — it never crashes. The agent saves its state (which tool was running, partial results), returns the interrupt to you, and waits. When you respond, the agent restores its state, re-runs the interrupted tool, and this time `interrupt()` returns your answer instead of raising an exception. The tool completes normally and the agent continues.

### Key Takeaways

1. **Two trigger points:** hooks (`BeforeToolCallEvent.interrupt()`) and tools (`ToolContext.interrupt()`)
2. **Dual-call design:** first call raises exception (pause), second call returns response (continue)
3. **`setdefault()` is the magic:** creates interrupt on first call, finds existing one on second call
4. **State is saved:** partial tool results are preserved so completed tools don't re-run
5. **Session-safe:** interrupt state can be serialized/deserialized across sessions

### Files to Read Next

| Priority | File | What you'll learn |
|----------|------|-------------------|
| 1 | `src/strands/interrupt.py` | Core classes (121 lines, very readable) |
| 2 | `src/strands/types/interrupt.py` | The `_Interruptible` protocol and the `interrupt()` method |
| 3 | `src/strands/event_loop/event_loop.py:460-505` | How the event loop handles interrupts |
| 4 | `tests_integ/interrupts/test_hook.py` | Integration test for hook-based interrupts |
| 5 | `tests_integ/interrupts/test_tool.py` | Integration test for tool-based interrupts |